#### Défi quotidien : Gestion et analyse des données en Python


Ce que vous apprendrez
Techniques avancées de normalisation, de réduction et d'agrégation des données.
Compétences en matière de collecte, d'exploration, d'intégration et de nettoyage de données à l'aide de Python.
Maîtrise de l'utilisation de Pandas pour la manipulation de données complexes.


Votre tâche
Téléchargez et importez l' ensemble de données sur les salaires des emplois en science des données .
Normalisez la colonne « salaire » en utilisant la normalisation Min-Max qui met à l'échelle toutes les valeurs de salaire entre 0 et 1.
Mettez en œuvre une réduction de dimensionnalité comme l'analyse en composantes principales (ACP) ou le t-SNE pour réduire le nombre de caractéristiques (colonnes) dans l'ensemble de données.
Regroupez l'ensemble de données par la colonne « niveau_d'expérience » et calculez le salaire moyen et médian pour chaque niveau d'expérience (par exemple, junior, intermédiaire, senior).
Indice :
Pour rappel, la normalisation est essentielle lorsqu'on traite des données présentant des plages de valeurs variables. Par exemple, les données salariales peuvent avoir une grande amplitude (de 20 000 $ à 200 000 $). En normalisant les données par la méthode Min-Max, on s'assure que toutes les valeurs salariales se situent dans une plage cohérente (de 0 à 1). Ceci est particulièrement utile lorsque les données sont destinées à être utilisées dans des modèles d'apprentissage automatique, car certains algorithmes (comme les k plus proches voisins ou les réseaux de neurones) sont plus performants lorsque les variables sont normalisées. Cela garantit qu'aucun salaire ne domine le processus d'apprentissage, rendant ainsi l'analyse plus équilibrée.

La réduction de dimensionnalité simplifie les ensembles de données complexes en diminuant le nombre de variables. Les données deviennent ainsi plus faciles à gérer, ce qui permet d'éviter le fléau de la dimensionnalité : un phénomène qui rend les modèles d'apprentissage automatique difficiles à traiter avec des données de grande dimension.
L'ACP, par exemple, permet de conserver les informations les plus importantes (la variance) de l'ensemble de données tout en réduisant le bruit et la redondance.
Elle peut également accélérer l'entraînement des modèles et faciliter la visualisation des données dans un espace de dimension réduite.

L'agrégation des données permet de comprendre les tendances au sein des sous-groupes de l'ensemble de données.
Le calcul des salaires moyens et médians pour chaque niveau d'expérience donne un aperçu de la répartition des rémunérations et des disparités entre les différents niveaux hiérarchiques. Ce type d'agrégation peut aider à répondre à des questions commerciales telles que : « Comment le salaire évolue-t-il avec l'expérience ? » ou « Quelle est la répartition des salaires pour les postes de direction ? »



Soumettez votre défi quotidien
Téléversez vos scripts Python, vos visualisations et un résumé de vos conclusions sur GitHub.

Une dernière chose : Bonne chance !

##


##

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

file_path = "dataset/datascience_salaries.csv"
if not os.path.exists(file_path):
    raise FileNotFoundError(f"Le fichier de données est introuvable : {file_path}")

# Chargement et nettoyage des colonnes
df = pd.read_csv(file_path, index_col=0)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print("Aperçu des données :")
print(df.head(), end="\n\n")

# Détection des colonnes salariale et de niveau d'expérience
salary_col = next((c for c in ["salaire", "salary", "annual_salary"] if c in df.columns), None)
experience_col = next((c for c in ["niveau_d_experience", "experience_level", "level"] if c in df.columns), None)

if salary_col is None:
    raise ValueError("Aucune colonne de salaire valide n'a été trouvée dans le jeu de données.")
if experience_col is None:
    raise ValueError("Aucune colonne de niveau d'expérience valide n'a été trouvée dans le jeu de données.")

# Conversion du salaire en nombre si nécessaire
if not pd.api.types.is_numeric_dtype(df[salary_col]):
    df[salary_col] = pd.to_numeric(df[salary_col], errors="coerce")

df = df.dropna(subset=[salary_col])

# Normalisation Min-Max de la colonne salaire
scaler = MinMaxScaler()
df["salaire_normalise"] = scaler.fit_transform(df[[salary_col]])
print("Salaires normalisés (Min-Max) :")
print(df[[salary_col, "salaire_normalise"]].head(), end="\n\n")

# Réduction de dimensionnalité avec l'ACP sur les colonnes numériques
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
if salary_col not in numeric_columns:
    numeric_columns.append(salary_col)

if len(numeric_columns) < 2:
    raise ValueError("Il n'y a pas suffisamment de colonnes numériques pour appliquer l'ACP.")

pca = PCA(n_components=min(2, len(numeric_columns)))
pca_result = pca.fit_transform(df[numeric_columns])
for i in range(pca_result.shape[1]):
    df[f"PCA_{i+1}"] = pca_result[:, i]

print("Résultats PCA :")
print(df[[f"PCA_{i+1}" for i in range(pca_result.shape[1])]].head(), end="\n\n")
print(f"Variance expliquée par les composantes : {pca.explained_variance_ratio_}", end="\n\n")

# Agrégation par niveau d'expérience
aggregation = df.groupby(experience_col)[salary_col].agg(["mean", "median"]).reset_index()
aggregation.columns = [experience_col, "salaire_moyen", "salaire_median"]

print("Salaire moyen et médian par niveau d'expérience :")
print(aggregation)